<a href="https://colab.research.google.com/github/Karolsak/advnaced-sieci/blob/claude%2Fpower-system-ode-solver-gui-011CV6Gg5ZcpsQ1MJkebuExe/Synchronous_Generator_Lab_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import math
import cmath
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox, AppLayout
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# --- Part 1: Solution to Example 7.3 ---

print("--- Solution to Example 7.3 ---")

# --- Given Nominal Parameters ---
Sn = 50e3       # Apparent power (VA)
V1Ln = 380.0    # Line-to-line voltage (V)
fn = 60.0       # Frequency (Hz)
nn = 1800.0     # Speed (rpm)
cos_phi_n = 0.82  # Power factor (lagging)
P_poles = (120 * fn) / nn  # Calculate number of poles

# --- Laboratory Test Data ---
# 1. Slip Test
Vs_slip = 115.0   # Applied line voltage for slip test (V)
Imax_slip = 22.2  # Maximum current (A)
Imin_slip = 11.3  # Minimum current (A)

# 2. Field Winding
Rf_base = 0.8     # Field resistance (Ohms)
T_base = 20.0     # Assumed base temperature for Rf (C)
alpha_cu = 0.00393 # Temp. coefficient for copper

# 3. No-Load Test
If0 = 10.5      # Field current for nominal no-load voltage (A)
Vn_phase = V1Ln / math.sqrt(3) # Nominal phase voltage (V)

# --- Assumptions ---
Ra = 0.0        # Stator resistance is neglected
T_op = 120.0    # Operating temperature for part (c) (C)

# --- (a) Find Synchronous Reactances Xsd and Xsq ---

print("\n--- Part (a): Synchronous Reactances ---")

Vph_slip = Vs_slip / math.sqrt(3)
print(f"Slip Test Phase Voltage (Vph_slip): {Vph_slip:.3f} V")

# Max impedance (Z_d) corresponds to min current (Imin)
# Min impedance (Z_q) corresponds to max current (Imax)
# Since Ra = 0, Z = X
Xsd = Vph_slip / Imin_slip
Xsq = Vph_slip / Imax_slip

print(f"Direct-Axis Synchronous Reactance (Xsd): {Xsd:.3f} Ohms")
print(f"Quad-Axis Synchronous Reactance (Xsq): {Xsq:.3f} Ohms")

# --- (b) Find Nominal Field Excitation Current Ifn ---

print("\n--- Part (b): Nominal Field Current ---")

# 1. Calculate nominal phase current
In = Sn / (math.sqrt(3) * V1Ln)
print(f"Nominal Phase Current (In): {In:.3f} A")

# 2. Calculate nominal power factor angle
phi_n = math.acos(cos_phi_n)
sin_phi_n = math.sin(phi_n)
print(f"Nominal Power Angle (phi_n): {math.degrees(phi_n):.3f} degrees")

# 3. Calculate the load angle (delta)
# Using the formula: tan(delta) = (In * Xsq * cos(phi)) / (Vn_phase + In * Xsq * sin(phi))
# (Note: sin(phi) is positive for lagging PF in this formulation)
numerator_delta = In * Xsq * cos_phi_n
denominator_delta = Vn_phase + In * sin_phi_n * Xsq # Corrected order based on typical Park's equations
tan_delta = numerator_delta / denominator_delta
delta = math.atan(tan_delta)
print(f"Nominal Load Angle (delta): {math.degrees(delta):.3f} degrees")

# 4. Calculate the internal no-load voltage (E_af)
# E_af = Vn_phase * cos(delta) + In * Xsd * sin(delta + phi)
# (Note: I_d = In * sin(delta + phi_n) )
E_af = Vn_phase * math.cos(delta) + In * math.sin(delta + phi_n) * Xsd
print(f"Internal No-Load Voltage (E_af): {E_af:.3f} V")

# 5. Calculate nominal field current (Ifn)
# Assuming linear magnetization curve
# Ifn / E_af = If0 / Vn_phase
Ifn = If0 * (E_af / Vn_phase)
print(f"Nominal Field Current (Ifn): {Ifn:.3f} A")

# --- (c) Find Voltage across Field Winding Terminals at 120 C ---

print("\n--- Part (c): Field Voltage at 120 C ---")

# 1. Calculate field resistance at 120 C
Rf_120 = Rf_base * (1 + alpha_cu * (T_op - T_base))
print(f"Field Resistance at {T_op} C (Rf_120): {Rf_120:.3f} Ohms")

# 2. Calculate field voltage
Vf_n = Ifn * Rf_120
print(f"Requested Field Voltage (Vf_n): {Vf_n:.3f} V")

print("\n--- End of Example 7.3 Solution ---")

# --- Part 2: Interactive Generator Lab ---

# Use the calculated values as defaults for the widgets
DEFAULT_Vt = Vn_phase
DEFAULT_If = Ifn
DEFAULT_P_mech = Sn * cos_phi_n # Default mechanical power = nominal real power
DEFAULT_cos_phi = cos_phi_n

# Create a central output widget for all text
log_output = widgets.Textarea(
    value='--- Interactive Lab Initialized ---\nWelcome. Adjust sliders and press [Run Simulation].\n',
    placeholder='Log output...',
    layout={'width': '100%', 'height': '200px'},
    disabled=False
)

def log_message(msg):
    """Appends a message to the log output."""
    log_output.value += f"{msg}\n"
    # Scroll to bottom
    log_output.scroll_top = 1000000

# --- Calculation Module (Steady-State) ---

def calculate_steady_state(Vt, P_mech, If, Xsd_val, Xsq_val, If0_val, Vt_n_val, Ra_val, Rf_val, T_op_val, T_base_val):
    """
    Recalculates the generator's steady-state operating point.
    This is a "what-if" calculator, not a dynamic ODE solver.

    Inputs:
    - Vt: Terminal phase voltage (V)
    - P_mech: Input mechanical power (W) (Assumed equal to output P_elec, losses neglected for P)
    - If: Field current (A)
    - Xsd_val, Xsq_val: Reactances (Ohms)
    - If0_val, Vt_n_val: No-load curve points (A, V)
    - Ra_val: Stator resistance (Ohms)
    - Rf_val, T_op_val, T_base_val: Thermal parameters

    Returns:
    - A dictionary of results or None on failure.
    """
    try:
        # 1. Calculate E_af from field current (linear mag curve)
        E_af = Vt_n_val * (If / If0_val)

        # 2. Solve for load angle (delta)
        # This is complex. P_elec = P(delta)
        # P_elec = (E_af * Vt / Xsd) * sin(delta) + (Vt^2 / 2) * (1/Xsq - 1/Xsd) * sin(2*delta)
        # This is a nonlinear equation. We need a numerical solver.
        # For simplicity, we'll iterate to find delta.

        delta_rad = math.radians(1.0) # Initial guess
        P_elec = 0.0

        # Use simple iterative solver (Newton-Raphson is better, but this is simpler)
        # We are looking for delta where P_calc(delta) = P_mech
        for i in range(500): # Increased iterations for better convergence
            P_term1 = (E_af * Vt / Xsd_val) * math.sin(delta_rad)
            P_term2 = (Vt**2 / 2.0) * (1/Xsq_val - 1/Xsd_val) * math.sin(2 * delta_rad)
            P_calc = 3.0 * (P_term1 + P_term2) # 3-phase power

            error = P_mech - P_calc

            if abs(error) < 1.0: # Convergence tolerance (1 Watt)
                break

            # Simple gradient ascent (not robust, but ok for demo)
            # Nudge delta in the direction that reduces error
            P_calc_plus = 3.0 * ((E_af * Vt / Xsd_val) * math.sin(delta_rad + 0.001) + \
                                (Vt**2 / 2.0) * (1/Xsq_val - 1/Xsd_val) * math.sin(2 * (delta_rad + 0.001)))
            derivative = (P_calc_plus - P_calc) / 0.001

            if derivative == 0:
                delta_rad += 0.001 # Avoid divide by zero
            else:
                delta_rad += error / derivative * 0.05 # Step, reduced learning rate to 0.05

            # Clamp delta_rad within physical limits after each step
            delta_rad = max(0.0, min(math.pi/2, delta_rad))

        else:
            log_message("Solver Warning: Failed to converge on delta after 500 iterations.")
            # return None

        # 3. Now that we have delta, find Id and Iq
        # Vq = Vt * cos(delta)
        # Vd = Vt * sin(delta)
        # Vq = E_af - Id*Xsd - Iq*Ra
        # Vd = -Iq*Xsq + Id*Ra

        # Vq = E_af - Id*Xsd (Ra=0)
        # Vd = -Iq*Xsq (Ra=0)

        # We must use Ra for a more general solution
        # System of 2 linear equations for Id, Iq:
        # Id * (Ra) + Iq * (Xsq) = Vt * sin(delta)
        # Id * (Xsd) + Iq * (Ra) = E_af - Vt * cos(delta)

        A = np.array([[Ra_val, Xsq_val], [Xsd_val, Ra_val]])
        B = np.array([Vt * math.sin(delta_rad), E_af - Vt * math.cos(delta_rad)])

        try:
            I_dq = np.linalg.solve(A, B)
            Id = I_dq[0]
            Iq = I_dq[1]
        except np.linalg.LinAlgError:
            log_message("Error: Matrix singularity. Check parameters (e.g., Ra=0).")
            return None

        # 4. Find complex Ia
        # Ia = Iq + jId (in d-q frame)
        # Rotate back to terminal frame (V is at 0 deg, E_af is at delta deg)
        # Ia_complex = (Iq + 1j * Id) * cmath.exp(1j * delta_rad)
        # Wait, the V-q, V-d equations were in the rotor (E_af) frame.
        # Let's re-do with V_t on the q-axis (easier)
        # This is a common point of confusion. Let's stick to the V_t @ 0 deg frame.
        # V_t = Vt + 0j
        # E_af_complex = E_af * cmath.exp(1j * delta_rad)
        # I_a_complex... this is getting complex.

        # Let's find Ia magnitude and angle from d-q components
        Ia_mag = math.sqrt(Id**2 + Iq**2)

        # Complex Power S = P + jQ
        # S = 3 * V_t * conj(I_a)
        # Let's find I_a (complex)
        # V_t_complex = Vt + 0j
        # E_af_complex = E_af * (math.cos(delta_rad) + 1j * math.sin(delta_rad))

        # V_t_complex - E_af_complex = -Z_d * I_d_vec - Z_q * I_q_vec
        # I_a_vec = I_d_vec + I_q_vec
        # This requires rotating frames.

        # Let's use a simpler, known power-angle relationship
        # P = 3 * ( (E_af*Vt/Xsd)*sin(delta) + (Vt^2/2)*(Xsd-Xsq)/(Xsd*Xsq)*sin(2*delta) )
        # Q = 3 * ( (E_af*Vt/Xsd)*cos(delta) - (Vt^2/Xsd) + (Vt^2/2)*(Xsd-Xsq)/(Xsd*Xsq)*cos(2*delta) )
        # This formula is slightly different. Let's re-check the P formula.

        # Let's use the Id, Iq we found.
        # P_out = 3 * (Vd*Id + Vq*Iq) -> This is power from machine
        # P_out = 3 * ( (Vt*sin(delta))*Id + (Vt*cos(delta))*Iq )
        # Q_out = 3 * (Vq*Id - Vd*Iq)
        # Q_out = 3 * ( (Vt*cos(delta))*Id - (Vt*sin(delta))*Iq )

        Vd = Vt * math.sin(delta_rad)
        Vq = Vt * math.cos(delta_rad)

        P_out_calc = 3 * (Vd * Id + Vq * Iq) # This should match P_mech (minus losses)
        Q_out_calc = 3 * (Vq * Id - Vd * Iq)

        S_out_calc = math.sqrt(P_out_calc**2 + Q_out_calc**2)
        Ia_calc = S_out_calc / (3 * Vt) if Vt > 0 else 0

        # 5. Loss Calculations
        P_cu_stator = 3.0 * (Ia_calc**2) * Ra_val

        # Field thermal calculation
        Rf_op = Rf_val * (1 + alpha_cu * (T_op_val - T_base_val))
        P_cu_field = (If**2) * Rf_op
        Vf_op = If * Rf_op

        # Assume Iron + Mech losses (P_core_mech)
        # This is a function of speed and voltage. Let's estimate it.
        # At no-load (If0), P_in = P_cu_field_0 + P_core_mech
        # Let's just assume a value for demo purposes.
        P_core_mech = 0.05 * Sn # Assume 5% of Sn

        P_loss_total = P_cu_stator + P_cu_field + P_core_mech

        # Final check on P_mech
        P_out_elec = P_out_calc
        P_in_mech = P_out_elec + P_cu_stator + P_core_mech # Power across air gap + losses

        # We set P_mech = P_in_mech
        # Our solver used P_mech = P_out_elec. This is a common simplification.
        # Let's stick with P_in_mech = P_mech from slider

        # P_out_elec = P_mech - P_cu_stator - P_core_mech
        # This makes the solver circular.
        # Let's assume P_mech is the target *electrical* output power P_out_elec

        P_out_elec = P_mech # Re-define slider meaning

        # We need to re-run the solver for delta with P_out_elec
        # The loop was already doing this. P_mech = P_calc

        P_in_mech = P_out_elec + P_cu_stator + P_core_mech

        # S_out
        S_out = P_out_elec + 1j * Q_out_calc
        S_mag = abs(S_out)
        Ia_mag = S_mag / (3 * Vt) if Vt > 0 else 0

        # Power Factor
        if P_out_elec == 0 and Q_out_calc == 0:
            cos_phi = 1.0
            pf_type = "No Load"
        elif P_out_elec == 0:
            cos_phi = 0.0
            pf_type = "Lagging" if Q_out_calc > 0 else "Leading"
        else:
            cos_phi = P_out_elec / S_mag
            pf_type = "Lagging" if Q_out_calc > 0 else "Leading"

        # Create result dictionary
        results = {
            "Vt": Vt,
            "If": If,
            "E_af": E_af,
            "delta": math.degrees(delta_rad),
            "P_out": P_out_elec,
            "Q_out": Q_out_calc,
            "S_out": S_mag,
            "Ia": Ia_mag,
            "cos_phi": cos_phi,
            "pf_type": pf_type,
            "Id": Id,
            "Iq": Iq,
            "P_in_mech": P_in_mech,
            "P_cu_stator": P_cu_stator,
            "P_cu_field": P_cu_field,
            "P_core_mech": P_core_mech,
            "P_loss_total": P_loss_total,
            "Efficiency": (P_out_elec / P_in_mech) * 100 if P_in_mech > 0 else 0,
            "Vf_op": Vf_op,
            "Rf_op": Rf_op,
            "Vt_complex": Vt + 0j,
            "E_af_complex": cmath.rect(E_af, delta_rad),
            "Ia_complex": (S_out / (3 * (Vt+0j))).conjugate() if Vt > 0 else 0j,
            "Xsd": Xsd_val,
            "Xsq": Xsq_val,
            "Ra": Ra_val
        }
        return results

    except Exception as e:
        log_message(f"Calculation Error: {e}")
        import traceback
        log_message(traceback.format_exc())
        return None


# --- UI Widgets ---

style = {'description_width': '150px'}
layout_auto = Layout(width='auto')

# --- Tab 1: Input Parameters ---
slider_Vt = widgets.FloatSlider(value=DEFAULT_Vt, min=0, max=DEFAULT_Vt * 1.5, step=1, description='Term. Phase Volt (Vt):', style=style, layout=layout_auto, readout_format='.1f')
slider_P_mech = widgets.FloatSlider(value=DEFAULT_P_mech, min=0, max=Sn * 1.2, step=100, description='Target Elec. Power (P):', style=style, layout=layout_auto, readout_format='.0f')
slider_If = widgets.FloatSlider(value=DEFAULT_If, min=0, max=DEFAULT_If * 2.0, step=0.1, description='Field Current (If):', style=style, layout=layout_auto, readout_format='.2f')

text_Xsd = widgets.FloatText(value=Xsd, description='Xsd (Ohms):', style=style, layout=layout_auto)
text_Xsq = widgets.FloatText(value=Xsq, description='Xsq (Ohms):', style=style, layout=layout_auto)
text_Ra = widgets.FloatText(value=0.01 * (Vn_phase/In), description='Ra (Ohms) (Assumed):', style=style, layout=layout_auto) # Assume 1%
text_If0 = widgets.FloatText(value=If0, description='If0 (no-load) (A):', style=style, layout=layout_auto)
text_Vn_phase = widgets.FloatText(value=Vn_phase, description='Vn_phase (no-load) (V):', style=style, layout=layout_auto)

tab_inputs = VBox([
    widgets.Label('Main Control Parameters:'),
    slider_Vt, slider_P_mech, slider_If,
    widgets.Label('Machine Model Parameters:'),
    HBox([VBox([text_Xsd, text_Xsq, text_Ra]), VBox([text_If0, text_Vn_phase])])
])

# --- Tab 2: Results & Visualization ---
output_plot = widgets.Output(layout={'width': '100%', 'height': '450px'})
output_text_results = widgets.Textarea(value='', placeholder='Results will appear here.', layout={'width': '100%', 'height': '200px'}, disabled=True)

def update_visualization(results):
    """Draws the phasor diagram."""
    with output_plot:
        clear_output(wait=True)
        if not results:
            print("No results to plot.")
            return

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.set_aspect('equal')

        # Get phasors
        V_t = results['Vt_complex']
        E_af = results['E_af_complex']
        I_a = results['Ia_complex']
        Ra = results['Ra']
        Xsd = results['Xsd']
        Xsq = results['Xsq']

        # Build other phasors
        V_Ra = Ra * I_a

        # d-q components of Ia
        delta_rad = math.radians(results['delta'])
        Ia_mag = abs(I_a)
        phi_rad = cmath.phase(I_a) # Angle of Ia

        # Angle of Ia relative to q-axis (E_af)
        relative_angle = phi_rad - delta_rad

        Id_mag = Ia_mag * math.sin(relative_angle)
        Iq_mag = Ia_mag * math.cos(relative_angle)

        # I_d is on d-axis, I_q is on q-axis
        # d-axis is at delta - 90
        # q-axis is at delta
        I_d_complex = cmath.rect(Id_mag, delta_rad - math.pi/2)
        I_q_complex = cmath.rect(Iq_mag, delta_rad)

        V_jIqXq = 1j * I_q_complex * Xsq
        V_jIdXd = 1j * I_d_complex * Xsd

        # V_t = E_af - V_Ra - V_jIdXd - V_jIqXq
        # (This is a check, let's plot E_af = V_t + V_Ra + ...)

        # Plot V_t
        ax.quiver(0, 0, V_t.real, V_t.imag, angles='xy', scale_units='xy', scale=1, color='b', label=f'Vt = {abs(V_t):.1f} V @ {cmath.phase(V_t):.1f} deg')

        # Plot E_af
        ax.quiver(0, 0, E_af.real, E_af.imag, angles='xy', scale_units='xy', scale=1, color='r', label=f'E_af = {abs(E_af):.1f} V @ {results["delta"]:.1f} deg')

        # Plot I_a (scaled)
        I_scale = abs(V_t) / abs(I_a) / 2 if abs(I_a) > 0 else 1
        ax.quiver(0, 0, I_a.real * I_scale, I_a.imag * I_scale, angles='xy', scale_units='xy', scale=1, color='g',
                  label=f'Ia = {abs(I_a):.1f} A @ {math.degrees(phi_rad):.1f} deg (scaled)')

        # Plot V_Ra
        ax.quiver(V_t.real, V_t.imag, V_Ra.real, V_Ra.imag, angles='xy', scale_units='xy', scale=1, color='c', linestyle='--')

        # Plot jIqXq
        ax.quiver( (V_t + V_Ra).real, (V_t + V_Ra).imag, V_jIqXq.real, V_jIqXq.imag, angles='xy', scale_units='xy', scale=1, color='m', linestyle='--')

        # Plot jIdXd
        ax.quiver( (V_t + V_Ra + V_jIqXq).real, (V_t + V_Ra + V_jIqXq).imag, V_jIdXd.real, V_jIdXd.imag, angles='xy', scale_units='xy', scale=1, color='orange', linestyle='--')

        # Check: E_af should be V_t + Ra*Ia + j*Iq*Xq + j*Id*Xd
        # Note: This is complex with frames. The plot shows V_t, E_af, and I_a.
        # Let's plot the d-q axes
        q_axis = E_af
        d_axis = cmath.rect(abs(E_af), delta_rad - math.pi/2)

        ax.quiver(0, 0, q_axis.real, q_axis.imag, angles='xy', scale_units='xy', scale=1, color='k', linestyle=':', label='q-axis')
        ax.quiver(0, 0, d_axis.real, d_axis.imag, angles='xy', scale_units='xy', scale=1, color='grey', linestyle=':', label='d-axis')

        max_val = max(abs(V_t), abs(E_af)) * 1.2
        ax.set_xlim(-max_val, max_val)
        ax.set_ylim(-max_val, max_val)
        ax.grid()
        ax.legend()
        ax.set_title('Steady-State Phasor Diagram')
        plt.xlabel('Real')
        plt.ylabel('Imaginary')
        plt.show()

def update_text_results(results):
    """Populates the text results box."""
    if not results:
        output_text_results.value = "Calculation failed. Check logs."
        return

    res_str = "--- STEADY-STATE RESULTS ---\n"
    res_str += f"  Operating Point:\n"
    res_str += f"    Term. Phase Voltage (Vt): {results['Vt']:.1f} V\n"
    res_str += f"    Field Current (If):       {results['If']:.2f} A\n"
    res_str += f"    Internal Voltage (E_af):  {results['E_af']:.1f} V\n"
    res_str += f"    Load Angle (delta):       {results['delta']:.2f} deg\n"
    res_str += "\n"
    res_str += "  Power & Current:\n"
    res_str += f"    Output Elec Power (P):    {results['P_out']/1000:.2f} kW\n"
    res_str += f"    Output Reactive Power (Q): {results['Q_out']/1000:.2f} kVAR\n"
    res_str += f"    Output Apparent Power (S): {results['S_out']/1000:.2f} kVA\n"
    res_str += f"    Stator Current (Ia):      {results['Ia']:.2f} A\n"
    res_str += f"    Power Factor (cos_phi):   {results['cos_phi']:.3f} {results['pf_type']}\n"
    res_str += "\n"
    res_str += "  d-q Components:\n"
    res_str += f"    Direct-axis Current (Id): {results['Id']:.2f} A\n"
    res_str += f"    Quad-axis Current (Iq):   {results['Iq']:.2f} A\n"

    output_text_results.value = res_str

tab_results = VBox([output_plot, output_text_results])


# --- Tab 3: Thermal, Losses & Economics ---
text_T_op = widgets.FloatText(value=T_op, description='Operating Temp (C):', style=style, layout=layout_auto)
text_T_base = widgets.FloatText(value=T_base, description='Base Rf Temp (C):', style=style, layout=layout_auto)
text_Rf_base = widgets.FloatText(value=Rf_base, description='Base Field R (Ohm):', style=style, layout=layout_auto)

text_cost_kwh = widgets.FloatText(value=0.15, description='Cost per kWh ($):', style=style, layout=layout_auto)
text_hours = widgets.FloatText(value=8, description='Operating Hours (h):', style=style, layout=layout_auto)

output_thermal_results = widgets.Textarea(value='', placeholder='Thermal results...', layout={'width': '100%', 'height': '150px'}, disabled=True)
output_econ_results = widgets.Textarea(value='', placeholder='Economic results...', layout={'width': '100%', 'height': '150px'}, disabled=True)

def update_thermal_econ_results(results):
    if not results:
        output_thermal_results.value = "N/A"
        output_econ_results.value = "N/A"
        return

    # Thermal
    therm_str = "--- THERMAL & FIELD WINDING ---\n"
    therm_str += f"  Operating Field Temp: {text_T_op.value:.1f} C\n"
    therm_str += f"  Operating Field R (Rf_op): {results['Rf_op']:.3f} Ohms\n"
    therm_str += f"  Operating Field V (Vf_op): {results['Vf_op']:.2f} V\n"
    output_thermal_results.value = therm_str

    # Losses
    loss_str = "--- LOSS BREAKDOWN ---\n"
    loss_str += f"  Input Mechanical Power: {results['P_in_mech']/1000:.3f} kW\n"
    loss_str += f"  Output Electrical Power: {results['P_out']/1000:.3f} kW\n"
    loss_str += f"  --------------------------------------\n"
    loss_str += f"  Total Losses:           {results['P_loss_total']/1000:.3f} kW\n"
    loss_str += f"    Stator Copper (I^2*Ra): {results['P_cu_stator']/1000:.3f} kW\n"
    loss_str += f"    Field Copper (If^2*Rf): {results['P_cu_field']/1000:.3f} kW\n"
    loss_str += f"    Core & Mech (Assumed):  {results['P_core_mech']/1000:.3f} kW\n"
    loss_str += f"  --------------------------------------\n"
    loss_str += f"  Efficiency:             {results['Efficiency']:.2f} %\n"

    # Economics
    cost_h = results['P_loss_total'] / 1000 * text_cost_kwh.value
    total_cost = cost_h * text_hours.value

    econ_str = "--- ECONOMIC ANALYSIS ---\n"
    econ_str += f"  Total Power Loss: {results['P_loss_total']/1000:.3f} kW\n"
    econ_str += f"  Cost of Losses (per hr): ${cost_h:.3f}\n"
    econ_str += f"  Cost for {text_hours.value} hours: ${total_cost:.2f}\n"

    output_thermal_results.value = therm_str + "\n" + loss_str
    output_econ_results.value = econ_str

tab_econ = VBox([
    widgets.Label('Thermal Parameters:'),
    HBox([text_T_op, text_T_base, text_Rf_base]),
    output_thermal_results,
    widgets.Label('Economic Parameters:'),
    HBox([text_cost_kwh, text_hours]),
    output_econ_results
])

# --- Tab 4: Advanced Simulation (Stub) ---
stub_text = """
--- DYNAMIC SIMULATION (Conceptual) ---

A full dynamic simulation (vs. this steady-state calculator)
requires solving a set of coupled Ordinary Differential Equations (ODEs)
in real-time, typically using a solver like RK45 (from scipy.integrate.solve_ivp).

1. STATE VARIABLES (y):
The state vector 'y' would include:
y = [i_d, i_q, i_f, i_kd, i_kq, omega, delta]
(d-q currents, field current, damper currents, speed, angle)

2. DIFFERENTIAL EQUATIONS (dy/dt):
You would need to define a function `def model(t, y)` that returns
the derivative `dy/dt` based on Park's Equations.

Example (simplified):
d(i_d)/dt = (1/L_d) * (v_d - R_a*i_d + omega*lambda_q)
d(omega)/dt = (1/J) * (T_m - T_e - B*omega)
T_e = (3/2)*(P/2)*(lambda_d*i_q - lambda_q*i_d)

3. PARAMETERS NEEDED:
This requires *many* more parameters not given in Example 7.3:
- L_d, L_q, L_f, L_ad, L_aq (self and mutual inductances)
- R_kd, R_kq (damper resistances)
- J (Inertia of generator + turbine)
- B (Damping coefficient)

4. ADVANCED CONTROLS:
- AVR (Voltage Regulator): A PID controller ODE would be
  added, calculating `Vf` based on `error = V_ref - V_t`.
- PSS (Power System Stabilizer): Another control loop
  that adjusts `Vf` to damp power oscillations.

5. MULTI-PHYSICS:
- Coupled Thermal Model: An ODE for temperature:
  d(T)/dt = (1/C_th) * (P_loss - (T - T_amb)/R_th)
- This `T` would update `Ra` and `Rf` inside the
  main model, coupling the physics.
"""
tab_adv_sim = VBox([widgets.Label("Notes on Dynamic & Multi-Physics Simulation:"),
                    widgets.Textarea(value=stub_text, layout={'width': '100%', 'height': '400px'}, disabled=True)])


# --- Main GUI Layout ---
tab_widget = widgets.Tab()
tab_widget.children = [tab_inputs, tab_results, tab_econ, tab_adv_sim]
tab_widget.set_title(0, 'Inputs')
tab_widget.set_title(1, 'Results & Phasor')
tab_widget.set_title(2, 'Losses & Economics')
tab_widget.set_title(3, 'Dynamic Sim Info')

# --- Control Buttons ---
button_run = widgets.Button(description='Run Simulation', button_style='success', icon='play')
button_reset = widgets.Button(description='Reset', button_style='info', icon='refresh')
button_stop = widgets.Button(description='Stop (N/A)', button_style='danger', icon='stop', disabled=True)

controls_box = HBox([button_run, button_reset, button_stop], layout=Layout(justify_content='center', padding='10px'))

# --- Main App Assembly ---
main_app = VBox([
    widgets.HTML("<h1>Salient-Pole Synchronous Generator Lab</h1>"),
    widgets.HTML(f"<i>Based on Example 7.3: {Sn/1000} kVA, {V1Ln} V, {fn} Hz, {P_poles:.0f} Poles</i>"),
    tab_widget,
    controls_box,
    log_output
], layout=Layout(width='100%', max_width='900px', margin='auto', border='1px solid #ccc', padding='10px'))

# --- Event Handlers ---
def on_run_button_clicked(b):
    """Main simulation callback."""
    log_message("\n--- Running Steady-State Calculation ---")

    # Get all input values
    params = {
        "Vt": slider_Vt.value,
        "P_mech": slider_P_mech.value,
        "If": slider_If.value,
        "Xsd_val": text_Xsd.value,
        "Xsq_val": text_Xsq.value,
        "If0_val": text_If0.value,
        "Vt_n_val": text_Vn_phase.value,
        "Ra_val": text_Ra.value,
        "Rf_val": text_Rf_base.value,
        "T_op_val": text_T_op.value,
        "T_base_val": text_T_base.value
    }

    # Run calculation
    results = calculate_steady_state(**params)

    if results:
        log_message("Calculation successful.")
        # Update all outputs
        update_text_results(results)
        update_visualization(results)
        update_thermal_econ_results(results)
    else:
        log_message("Calculation failed. Check parameters and log.")

def on_reset_button_clicked(b):
    """Resets all inputs to their default values."""
    log_message("\n--- Resetting to Nominal Values ---")
    slider_Vt.value = DEFAULT_Vt
    slider_P_mech.value = DEFAULT_P_mech
    slider_If.value = DEFAULT_If

    text_Xsd.value = Xsd
    text_Xsq.value = Xsq
    text_Ra.value = 0.01 * (Vn_phase/In) # Re-calc assumed Ra
    text_If0.value = If0
    text_Vn_phase.value = Vn_phase

    text_T_op.value = T_op
    text_T_base.value = T_base
    text_Rf_base.value = Rf_base

    text_cost_kwh.value = 0.15
    text_hours.value = 8

    # Clear outputs
    with output_plot:
        clear_output()
    output_text_results.value = "Reset to defaults."
    output_thermal_results.value = ""
    output_econ_results.value = ""

# Link buttons to functions
button_run.on_click(on_run_button_clicked)
button_reset.on_click(on_reset_button_clicked)

# --- Final Display ---
print("\n--- Interactive Lab ---")
print("Displaying the interactive lab interface. (If this is not a notebook, the GUI will not appear.)")

# Display the main application
display(main_app)

# Automatically run the simulation with default values
on_run_button_clicked(None)

--- Solution to Example 7.3 ---

--- Part (a): Synchronous Reactances ---
Slip Test Phase Voltage (Vph_slip): 66.395 V
Direct-Axis Synchronous Reactance (Xsd): 5.876 Ohms
Quad-Axis Synchronous Reactance (Xsq): 2.991 Ohms

--- Part (b): Nominal Field Current ---
Nominal Phase Current (In): 75.967 A
Nominal Power Angle (phi_n): 34.915 degrees
Nominal Load Angle (delta): 28.065 degrees
Internal No-Load Voltage (E_af): 591.234 V
Nominal Field Current (Ifn): 28.296 A

--- Part (c): Field Voltage at 120 C ---
Field Resistance at 120.0 C (Rf_120): 1.114 Ohms
Requested Field Voltage (Vf_n): 31.533 V

--- End of Example 7.3 Solution ---

--- Interactive Lab ---
Displaying the interactive lab interface. (If this is not a notebook, the GUI will not appear.)


In [8]:
slider_If.value = 30.0 # Adjust Field Current to 30 A
on_run_button_clicked(None)
